### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="sdss_17",
    dataset_year="2022",
    domain_str="physics & astronomy",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/datasets/fedesoriano/stellar-classification-dataset-sdss17",
    download_description="""
We download the data from Kaggle.

mkdir -p local-data-warehouse/sdss_17/ && cd local-data-warehouse/sdss_17 && kaggle datasets download fedesoriano/stellar-classification-dataset-sdss17 && cd ../../ && unzip local-data-warehouse/sdss_17/stellar-classification-dataset-sdss17.zip -d local-data-warehouse/sdss_17 && rm local-data-warehouse/sdss_17/stellar-classification-dataset-sdss17.zip
""",
    # References
    academic_reference_bibtex=r"""@article{accetta2022seventeenth,
  title={The seventeenth data release of the Sloan Digital Sky Surveys: Complete release of MaNGA, MaStar, and APOGEE-2 data},
  author={Accetta, Katherine and Aerts, Conny and Aguirre, Victor Silva and Ahumada, Romina and Ajgaonkar, Nikhil and Ak, N Filiz and Alam, Shadab and Prieto, Carlos Allende and Almeida, Andres and Anders, Friedrich and others},
  journal={The Astrophysical Journal Supplement Series},
  volume={259},
  number={2},
  pages={35},
  year={2022},
  publisher={IOP Publishing}
}
""",
    academic_reference_bibtex_key="accetta2022seventeenth",
    license="Public Domain", # On Kaggle it says Data files © Original Authors which is under public domain.
    data_tags=["IID"],
    curation_comments="""
- We renamed the target feature.
- We dropped duplicates based on "obj_ID" to avoid target leakage from subgroups. 
- We dropped several (ID-like) meta-features that seem to be not part of the predictive task.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="ObjectType",
    problem_type="multiclass_classification",
    objective_metric_name="log_loss",
    stratify_on="ObjectType",
)

## Preprocessing

In [2]:
import pandas as pd

df = pd.read_csv(f"{dataset_mold.path}/star_classification.csv")

target_feature = "ObjectType"
df = df.rename(columns={"class": target_feature})

df = df.drop_duplicates(subset=["obj_ID"])

# Note: the following is from a very naive domain knowledge perspective
# and should be checked with domain experts. But otherwise, the predictive task
# might leak. So we are better safe than sorry.
df = df.drop(
    columns=[
        "obj_ID",  # should not be predictive, also has duplicates?
        "spec_obj_ID",  # ID
        "run_ID",  # should not be predictive but might indicate confounding noise
        "rerun_ID",  # constant
        "field_ID",  # might indicate clusters/subgroups of data?
        "MJD",  # date of observation, should not be predictive?
    ]
)

# Data is ordered, thus dist shift for original order. Shuffling the data removes this.
df = df.sample(frac=1, random_state=42).reset_index(drop=True)


cat_features = [
    "ObjectType",
    "plate",
    "fiber_ID",
]
df[cat_features] = df[cat_features].astype("category")

## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 78,053
Columns: 12
Use sampling: False (sample size: 78,053)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['alpha', 'delta', 'redshift', 'u', 'g', 'i', 'r', 'z', 'plate', 'fiber_ID']
Rows remaining as candidates after top-10 filter: 0 (of 78,053)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,alpha,delta,u,g,r,i,z,cam_col,ObjectType,redshift,plate,fiber_ID
0,254.944575,21.156599,18.18506,16.91576,17.08241,17.21398,17.27512,5,STAR,-0.000942,3290,126
1,219.212586,2.301101,23.20924,21.60087,20.00037,19.23079,18.53441,4,GALAXY,0.445265,4021,567
2,168.550571,10.141066,20.21758,18.87304,18.30956,18.08647,17.99164,5,STAR,0.000381,2413,475
3,188.612861,5.531677,19.02684,17.64901,16.97685,16.65568,16.42887,5,GALAXY,0.081323,845,551
4,193.617099,6.202520,20.02678,18.23957,17.34360,16.94755,16.63911,1,GALAXY,0.079594,1792,128


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,ObjectType,category,0.0,0.0,3.0,"GALAXY, STAR, QSO"
1,plate,category,0.0,0.0,6264.0,"7147, 7450, 4095, 6301, 5061, 11317, 3656, 5466, 7150, 7697"
2,fiber_ID,category,0.0,0.0,1000.0,"597, 637, 105, 599, 475, 333, 621, 391, 189, 481"
3,alpha,float64,0.0,0.0,78053.0,"339.6263, 254.9446, 219.2126, 168.5506, 188.6129, 6.4067, 137.6021, 37.2564, 224.2434, 349.2097"
4,delta,float64,0.0,0.0,78053.0,"27.4648, 21.1566, 2.3011, 10.1411, 5.5317, 19.7955, 53.647, -4.3247, 37.5812, 3.5911"
5,u,float64,0.0,0.0,74160.0,"24.6347, 24.6347, 24.6347, 24.6346, 24.6347, 22.5, 20.2601, 24.6346, 25.4147, 19.1728"
6,g,float64,0.0,0.0,73479.0,"25.1144, 25.1144, 25.1144, 20.4269, 22.4501, 21.7641, 21.9479, 21.6514, 22.0831, 21.8285"
7,r,float64,0.0,0.0,72978.0,"24.802, 24.802, 24.802, 20.467, 21.4462, 21.4976, 20.8977, 20.5484, 20.2045, 20.7653"
8,i,float64,0.0,0.0,73029.0,"24.3618, 24.3618, 19.7357, 20.234, 19.8325, 19.6294, 17.3595, 20.3513, 19.9581, 19.6762"
9,z,float64,0.0,0.0,72933.0,"22.8269, 22.8269, 22.8269, 22.8269, 19.9222, 19.8872, 19.2691, 19.5089, 19.8728, 18.8939"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
alpha,78053.0,179.903175,95.575830,0.005528,359.999810
delta,78053.0,24.145442,19.407268,-18.785328,83.000519
u,78053.0,22.031628,35.941238,-9999.000000,30.660390
g,78053.0,20.562950,35.922825,-9999.000000,31.602240
r,78053.0,19.677898,1.857740,9.822070,29.571860
i,78053.0,19.097805,1.749224,9.469903,30.250090
z,78053.0,18.644319,35.900004,-9999.000000,28.238290
cam_col,78053.0,3.471974,1.587801,1.000000,6.000000
redshift,78053.0,0.572657,0.723595,-0.009971,7.011245


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column     rank                      
ObjectType 1     GALAXY  47613  61.00
           2       STAR  16446  21.07
           3        QSO  13994  17.93
fiber_ID   1        597    134   0.17
           2        637    127   0.16
           3        105    122   0.16
           4        599    122   0.16
           5        475    120   0.15
plate      1       7147     73   0.09
           2       7450     70   0.09
           3       4095     68   0.09
           4       6301     61   0.08
           5       5061     61   0.08

In [8]:
# Target Distribution
target_df

,count,pct
ObjectType,,
GALAXY,47613,61.00
STAR,16446,21.07
QSO,13994,17.93


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=3, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...


Saving curated container to sdss_17/019d7369-942d-72c0-b327-e812c4b56b7e


019d7369-942d-72c0-b327-e812c4b56b7e
f1a03ae52a688c1f4c42335fd3cf823b9f661a3e0901b5188b5d16a062d2315a
